# Reading -> Transforming -> Writing (from : bronze.px_cat_g1v2_raw to silver.erp_px_cat_g1v2)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Reading table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2_raw")
df.printSchema()

In [0]:
df.count()


In [0]:
df.display()

## Basic cleanup and transformation

### Changing column name

In [0]:
df = df.withColumnRenamed("CAT", "CATEGORY")\
        .withColumnRenamed("SUBCAT", "SUBCATEGORY")
        

### Trimming spaces

In [0]:
df.printSchema()

In [0]:
for field_name in df.schema.fieldNames():
    df = df.withColumn(field_name, trim(col(field_name)))


In [0]:
df.display()

## Validation of data

### Incorrect values in Maintainence

In [0]:
df.groupBy("MAINTENANCE").count().display()

### Incorrect values in Subcategory

In [0]:
df.groupBy("SUBCATEGORY").count().distinct().display()

### Incorrect value in Category

In [0]:
df.groupBy("CATEGORY").count().distinct().display()

In [0]:
df.groupBy("SUBCATEGORY", "CATEGORY").count().display()

In [0]:
print(f"subcategory count : {df.select("SUBCATEGORY").count()}")
print(f"subcategory distinct count : {df.select("SUBCATEGORY").distinct().count()}")


### Duplicate CID

In [0]:
print(f"ID count : {df.select("ID").count()}")
print(f"ID distinct count : {df.select("ID").distinct().count()}")

## Writing data 

In [0]:
df.write\
.mode("overwrite")\
.format("delta")\
.saveAsTable("workspace.silver.erp_px_cat_g1v2")